## Spark Streaming and Using PostgreSQL as a Sink

### Introduction to Spark Streaming Sinks
In Spark Streaming, a sink is where the output of the stream processing is sent. The choice of sink depends on the system requirements and the use case. Common sinks include databases, files, and real-time dashboards. 

Here, we focus on using PostgreSQL as a sink.

### Setting Up Spark with JDBC for PostgreSQL
To use PostgreSQL as a sink in Spark Streaming, you need to configure the Spark session with the PostgreSQL JDBC driver. This enables Spark to send data directly to your PostgreSQL database.


### Defining Connection Properties
Define the necessary connection properties to connect to your PostgreSQL database:

```python
# Connection URL and properties for PostgreSQL
url = "jdbc:postgresql://soe-postgres.postgres.database.azure.com:5432/music_store"
postgres_options = {
    "url": url,
    "user": "student",
    "password": "reRZ2pjg1WxqlwjU",
    "driver": "org.postgresql.Driver"
}
```

Like before we'll provide the path to postgreSQL library.

In [2]:
import os

# Path to the PostgreSQL JDBC jar file
rel_path = os.path.relpath('./libs/postgresql-42.7.3.jar')

# Connection URL and properties for PostgreSQL
url = "jdbc:postgresql://fhtw-big-data.postgres.database.azure.com/music_store"
postgres_options = {
    "url": url,
    "user": "student",
    "password": "reRZ2pjg1WxqlwjU",
    "driver": "org.postgresql.Driver"
}

In [3]:
# Kafka configuration parameters for Confluent Cloud
kafka_params = {
      "kafka.bootstrap.servers": "46.225.20.89:9092",
      "subscribe": "fhtw-stocks",
      "kafka.security.protocol": "PLAINTEXT",
      "startingOffsets": "latest",
}

### Configure your spark session to include custom jar files

In Apache Spark, external libraries, such as JDBC drivers or connectors for other data sources like Apache Kafka, can be included in your Spark jobs by configuring the Spark session

In [4]:
import os
from pyspark.sql import SparkSession

# Create a Spark session and configure it to use the JDBC jar
spark = SparkSession.builder \
    .appName("PostgreSQL Integration Example") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0") \
    .config("spark.jars", rel_path) \
    .config("spark.executor.extraClassPath", rel_path) \
    .config("spark.driver.extraClassPath", rel_path) \
    .getOrCreate()

26/06/08 18:08:04 WARN Utils: Your hostname, codespaces-f5202d resolves to a loopback address: 127.0.0.1; using 10.0.0.231 instead (on interface eth0)
26/06/08 18:08:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/sdkman/candidates/spark/3.5.1/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vscode/.ivy2/cache
The jars for the packages stored in: /home/vscode/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f78f8e59-93ca-404b-900f-737b87b2bf75;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.9.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 732ms :: artifacts dl 42ms
	:: 

### Data Structure

The messages in the Kafka topic have the following JSON structure:

```json
{
  "side": "SELL",
  "quantity": 1587,
  "symbol": "ZVV",
  "price": 326,
  "account": "LMN456",
  "userid": "User_5"
}


## Streaming Data Processing and Writing to PostgreSQL
Once the Spark session is set up and configured, you can process streaming data and write the output directly to PostgreSQL.

### Reading from Kafka
Assuming data is being streamed from Kafka, set up the stream reading:

In [5]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

# Define the schema for the JSON data
schema = StructType([
    StructField("side", StringType(), nullable=False),
    StructField("quantity", IntegerType(), nullable=False),
    StructField("symbol", StringType(), nullable=False),
    StructField("price", FloatType(), nullable=False),
    StructField("account", StringType(), nullable=False),
    StructField("userid", StringType(), nullable=False)
])

In [6]:
# Read streaming data from Kafka
df = spark \
    .readStream \
    .format("kafka") \
    .options(**kafka_params) \
    .load()

# Select and cast the key and value columns to STRING
df_message = df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")

# Parse the JSON data and apply schema
df_value = df_message.selectExpr("CAST(value AS STRING) as json_string")

parsed_df = df_value.withColumn("jsonData", 
                                from_json(col("json_string"), schema)).select("jsonData.*")

In [7]:
# Display the total quantity
query = parsed_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

import time
time.sleep(30)
query.stop()

26/06/08 18:08:32 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-6229771c-5c44-46e5-af5a-e3745cd5a2f8. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/08 18:08:32 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/08 18:08:33 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 0
-------------------------------------------
+----+--------+------+-----+-------+------+
|side|quantity|symbol|price|account|userid|
+----+--------+------+-----+-------+------+
+----+--------+------+-----+-------+------+



-------------------------------------------
Batch: 1
-------------------------------------------
+----+--------+------+-------+-------+------+
|side|quantity|symbol|  price|account|userid|
+----+--------+------+-------+-------+------+
|SELL|     747|  AAPL| 795.08| ACC123|User_4|
| BUY|     467|  NFLX|1005.88| ACC123|User_5|
+----+--------+------+-------+-------+------+

-------------------------------------------
Batch: 2
-------------------------------------------
+----+--------+------+-------+-------+------+
|side|quantity|symbol|  price|account|userid|
+----+--------+------+-------+-------+------+
|SELL|     801|  AAPL|  349.2| ACC123|User_3|
|SELL|     891|  AAPL|1277.29| ACC345|User_2|
| BUY|     661|  MSFT|1118.44| ACC789|User_1|
+----+--------+------+-------+-------+------+

-------------------------------------------
Batch: 3
-------------------------------------------
+----+--------+------+-------+-------+------+
|side|quantity|symbol|  price|account|userid|
+----+--------+--

-------------------------------------------
Batch: 4
-------------------------------------------
+----+--------+------+------+-------+------+
|side|quantity|symbol| price|account|userid|
+----+--------+------+------+-------+------+
| BUY|     358|  MSFT|233.14| ACC789|User_5|
+----+--------+------+------+-------+------+

-------------------------------------------
Batch: 5
-------------------------------------------
+----+--------+------+-------+-------+------+
|side|quantity|symbol|  price|account|userid|
+----+--------+------+-------+-------+------+
| BUY|     509|  MSFT|1921.27| ACC345|User_1|
+----+--------+------+-------+-------+------+



-------------------------------------------
Batch: 6
-------------------------------------------
+----+--------+------+------+-------+------+
|side|quantity|symbol| price|account|userid|
+----+--------+------+------+-------+------+
| BUY|     326|  MSFT|470.26| ACC456|User_4|
+----+--------+------+------+-------+------+



...

-------------------------------------------
Batch: 13
-------------------------------------------
+----+--------+------+------+-------+------+
|side|quantity|symbol| price|account|userid|
+----+--------+------+------+-------+------+
|SELL|     567|  AAPL|267.96| ACC345|User_3|
+----+--------+------+------+-------+------+

-------------------------------------------
Batch: 14
-------------------------------------------
+----+--------+------+------+-------+------+
|side|quantity|symbol| price|account|userid|
+----+--------+------+------+-------+------+
|SELL|     521|  AAPL|247.16| ACC012|User_5|
+----+--------+------+------+-------+------+



-------------------------------------------
Batch: 15
-------------------------------------------
+----+--------+------+-------+-------+------+
|side|quantity|symbol|  price|account|userid|
+----+--------+------+-------+-------+------+
| BUY|     652|  MSFT|1663.05| ACC012|User_2|
+----+--------+------+-------+-------+------+

-------------------------------------------
Batch: 16
-------------------------------------------
+----+--------+------+-------+-------+------+
|side|quantity|symbol|  price|account|userid|
+----+--------+------+-------+-------+------+
|SELL|     845|  AAPL|1562.16| ACC012|User_5|
+----+--------+------+-------+-------+------+

-------------------------------------------
Batch: 17
-------------------------------------------
+----+--------+------+------+-------+------+
|side|quantity|symbol| price|account|userid|
+----+--------+------+------+-------+------+
|SELL|     866|  AAPL|302.76| ACC456|User_5|
+----+--------+------+------+-------+------+

------------------

-------------------------------------------
Batch: 22
-------------------------------------------
+----+--------+------+-------+-------+------+
|side|quantity|symbol|  price|account|userid|
+----+--------+------+-------+-------+------+
|SELL|     730|  AAPL|1166.09| ACC456|User_1|
+----+--------+------+-------+-------+------+



-------------------------------------------
Batch: 23
-------------------------------------------
+----+--------+------+------+-------+------+
|side|quantity|symbol| price|account|userid|
+----+--------+------+------+-------+------+
|SELL|     620|  AAPL|287.78| ACC123|User_1|
+----+--------+------+------+-------+------+



-------------------------------------------
Batch: 24
-------------------------------------------
+----+--------+------+-------+-------+------+
|side|quantity|symbol|  price|account|userid|
+----+--------+------+-------+-------+------+
|SELL|     711|  AAPL|1708.89| ACC123|User_2|
+----+--------+------+-------+-------+------+

-------------------------------------------
Batch: 25
-------------------------------------------
+----+--------+------+-------+-------+------+
|side|quantity|symbol|  price|account|userid|
+----+--------+------+-------+-------+------+
|SELL|     690|  AAPL|1772.27| ACC345|User_1|
+----+--------+------+-------+-------+------+



### Understanding `foreachBatch` in Spark Streaming

#### Introduction to `foreachBatch`
In Spark Structured Streaming, `foreachBatch()` is a method that allows you to perform custom processing on the output of each micro-batch of a streaming DataFrame/Dataset. It provides the flexibility to apply batch DataFrame operations that are not available in the standard streaming DataFrame operations.

#### How `foreachBatch` Works
`foreachBatch()` takes a function as an argument. This function should have two parameters: a DataFrame representing the batch data and a long type batch ID, which is unique for each micro-batch. Inside this function, you can perform any operation that is available for batch DataFrames, such as writing to databases, performing complex transformations, or even generating plots for visualization.

##### Example Usage
Here’s how you can use `foreachBatch()` to write streaming data into a PostgreSQL database:

```python
from pyspark.sql import DataFrame

def write_to_postgres(df: DataFrame, epoch_id: int):
    df.write.format("jdbc") \
        .option("url", "jdbc:postgresql://database_url:5432/database_name") \
        .option("dbtable", "target_table") \
        .option("user", "username") \
        .option("password", "password") \
        .option("driver", "org.postgresql.Driver") \
        .mode("append") \
        .save()

# Assuming 'streaming_df' is your streaming DataFrame
query = streaming_df.writeStream \
    .foreachBatch(write_to_postgres) \
    .start()

query.awaitTermination()

In [ ]:
from pyspark.sql.functions import sum

# writing each batch result to the console

def write_to_console(df, epoch_id):
    display(df.toPandas())

# Write results to PostgreSQL using foreachBatch
query = parsed_df.writeStream \
    .outputMode("append") \
    .foreachBatch(write_to_console) \
    .start()

time.sleep(15)
query.stop()

26/06/08 18:11:04 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-1ebc93cb-3f9d-4475-a01b-27fa7791d754. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/08 18:11:04 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/08 18:11:04 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


,side,quantity,symbol,price,account,userid
0,BUY,665,MSFT,1799.73999,ACC456,User_5


,side,quantity,symbol,price,account,userid
0,BUY,643,MSFT,303.220001,ACC123,User_3
1,SELL,832,AAPL,1207.380005,ACC789,User_4
2,SELL,936,AAPL,1488.469971,ACC456,User_2
3,BUY,434,MSFT,378.540009,ACC456,User_1
4,SELL,855,AAPL,1393.550049,ACC345,User_4
5,SELL,558,AAPL,874.289978,ACC345,User_2
6,BUY,471,MSFT,1318.780029,ACC789,User_1
7,BUY,311,TSLA,1490.839966,ACC345,User_1
8,SELL,891,AAPL,850.349976,ACC456,User_3
9,SELL,726,AAPL,208.440002,ACC456,User_3


,side,quantity,symbol,price,account,userid
0,BUY,360,MSFT,1635.569946,ACC456,User_3


,side,quantity,symbol,price,account,userid
0,SELL,713,AAPL,1953.550049,ACC789,User_2


26/06/08 18:11:19 WARN TaskSetManager: Lost task 0.0 in stage 30.0 (TID 30) (4e94739b-8d21-4ff0-ad7e-1707a4d99a2d.internal.cloudapp.net executor driver): TaskKilled (Stage cancelled: Job 30 cancelled part of cancelled job group 68bc1902-a678-4c69-8342-0aa2fb180606)


For storing the data in postgreSQL now, all we need to do, is to stream into the database and configure the jdbc sink 

In [9]:
# Write results to PostgreSQL using foreachBatch

student_id = "ds25m046" # Replace with your actual student ID!


table_name = f"streaming_data.{student_id}_stock_data"

def write_to_postgres(df, epoch_id):
    df.write \
        .format("jdbc") \
        .options(**postgres_options) \
        .option("dbtable", table_name) \
        .mode("append") \
        .save()

#### After running the next cell, look inside the database into your table using streaming_data, you should see entries being added!

In [10]:
from pyspark.sql.functions import sum

query = parsed_df.writeStream \
    .outputMode("append") \
    .foreachBatch(write_to_postgres) \
    .start()

26/06/08 18:11:48 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-abe9027e-a719-476e-897d-e1f5a5a2c707. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/08 18:11:48 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/06/08 18:11:48 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


In [ ]:
# ALWAYS stop a stream otherwise you might flood the database
query.stop()

### Calculate the total quantity of stocks traded and store in database

#### After running the next cell, look inside the database into your table using streaming_data, you should see entries being added!

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "4")

In [12]:
# Write results to PostgreSQL using foreachBatch
student_id = "YOUR STUDENT ID" # Replace with your actual student ID!
table_name = f"streaming_data.{student_id}_stock_data_quantity"

# Write results to PostgreSQL using foreachBatch
def write_to_postgres(df, epoch_id):
    df.write \
        .format("jdbc") \
        .options(**postgres_options) \
        .option("dbtable", table_name) \
        .mode("append") \
        .save()

# Calculate the total quantity of stocks traded
symbol_count_df = (parsed_df
                   .groupBy("symbol")
                   .count()
                   .withColumnRenamed("count", "trade_count"))

print(f'Starting to write to table {table_name}')

# Display the count of trades for each symbol
query = symbol_count_df.writeStream \
    .outputMode("complete") \
    .foreachBatch(write_to_postgres) \
    .start()    

Starting to write to table streaming_data.YOUR STUDENT ID_stock_data_quantity


26/06/08 18:14:20 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-d1c0065c-886d-49c4-84e9-bb5b58762c8a. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/08 18:14:20 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/06/08 18:14:20 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/06/08 18:14:21 ERROR MicroBatchExecution: Query [id = bdacce89-193f-4873-b966-0c7e9f890ba6, runId = f1e081fe-d421-4e2c-9d13-98d91eb5c453] terminated with error
py4j.Py4JException: An exception was raised by the Python Proxy. Return Message: Traceback (most recent call last):
  File "/home/vscode/.local/lib/python3.11/site-packages/py4j/clientserver.py", line 617, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/vscode/.local/lib/python3.11/site-packages/pyspark/sql/utils.py", line 115, in call
    raise e
  File "/home/vscode/.local/lib/python3.11/site-packages/pyspark/sql/utils.py", line 112, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
  File "/tmp/ipykernel_3

In [13]:
query.stop()

#### foreachBatch parameters
Typically, foreachBatch expects a function with exactly two arguments: the DataFrame and the epoch ID. To pass additional parameters, you can use a lambda or define a higher-order function that returns the desired function with additional parameters.

In [14]:
def write_to_postgres(table_name):
    def inner_write_to_postgres(df, epoch_id):
        df.write \
            .format("jdbc") \
            .options(**postgres_options) \
            .option("dbtable", table_name) \
            .mode("append") \
            .save()
    return inner_write_to_postgres

In [15]:
from pyspark.sql.functions import current_timestamp

student_id = "YOUR STUDENT ID" # Replace with your actual student ID!
table_name = f"streaming_data.{student_id}_stock_data_quantity_timestamp"

parsed_df_with_time = parsed_df.withColumn("timestamp", current_timestamp())

# Group by symbol and timestamp, then count
symbol_count_df = (parsed_df_with_time
                   .groupBy("symbol", "timestamp")
                   .count()
                   .withColumnRenamed("count", "trade_count"))

print(f'Starting to write to table {table_name}')

# Continue with your streaming write
query = symbol_count_df.writeStream \
    .outputMode("complete") \
    .foreachBatch(write_to_postgres(table_name)) \
    .start()

Starting to write to table streaming_data.YOUR STUDENT ID_stock_data_quantity_timestamp


26/06/08 18:14:35 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-6da2854d-5af3-4f96-b861-41843fa78215. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/08 18:14:35 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/06/08 18:14:35 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/06/08 18:14:36 ERROR MicroBatchExecution: Query [id = c3b31ecb-1cf4-47dd-80cb-883547a9e9a4, runId = 71575b2f-eca2-46f1-85af-c73df8e23d14] terminated with error
py4j.Py4JException: An exception was raised by the Python Proxy. Return Message: Traceback (most recent call last):
  File "/home/vscode/.local/lib/python3.11/site-packages/py4j/clientserver.py", line 617, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/vscode/.local/lib/python3.11/site-packages/pyspark/sql/utils.py", line 115, in call
    raise e
  File "/home/vscode/.local/lib/python3.11/site-packages/pyspark/sql/utils.py", line 112, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
  File "/tmp/ipykernel_3

In [16]:
query.stop()

In [17]:
# stop all running spark streams
for s in spark.streams.active:
    s.stop()

In [18]:
spark.stop()